In [ ]:
"Shor's Algorithm - Scaling Analysis for Classical Period Finding"
import math
import os
import numpy as np
import time
import random
import pandas as pd

def gcd(a, b):
    """Euclidean algorithm for greatest common divisor."""
    while b:
        a, b = b, a % b
    return a

def is_prime(n):
    """Trial division primality test."""
    if n < 2 or (n > 2 and n % 2 == 0):
        return False
    if n == 2:
        return True
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def factorize(N):
    """Return list of prime factors (trial division)."""
    factors = []
    d, temp = 2, N
    while d * d <= temp:
        while temp % d == 0:
            factors.append(d)
            temp //= d
        d += 1
    if temp > 1:
        factors.append(temp)
    return factors

def is_semiprime(n):
    """True if n = p × q for distinct primes (RSA-style)."""
    f = factorize(n)
    return len(f) == 2 and f[0] != f[1]

def classical_order(x, N):
    """Find smallest r with x^r ≡ 1 (mod N). Returns (r, op_count)."""
    r, value, ops = 1, x % N, 0
    while value != 1:
        value = (value * x) % N
        r, ops = r + 1, ops + 1
        if r > N:
            return None, ops
    return r, ops

def analyze_single_iteration(N, a):
    """One Shor iteration: pick base a, find order, extract factors if possible."""
    ops = 1
    if gcd(a, N) != 1:  # Lucky: a shares factor with N
        return {'success': True, 'operations': ops}
    r, o = classical_order(a, N)
    ops += o
    if r is None or r % 2 != 0:  # Need even order
        return {'success': False, 'operations': ops}
    x_half = pow(a, r // 2, N)
    if x_half == N - 1:  # x^(r/2) ≡ -1: no useful factors
        return {'success': False, 'operations': ops + 1}
    p, q = gcd(x_half - 1, N), gcd(x_half + 1, N)  # Extract factors
    success = (1 < p < N) or (1 < q < N)
    return {'success': success, 'operations': ops + 3}

def run_iterations(N, num_iterations=10):
    """Run Shor iterations with random coprime bases; return stats."""
    bases = [a for a in range(2, min(N, 10000)) if gcd(a, N) == 1]
    if not bases:
        return None
    
    # Warm-up run to avoid JIT/startup overhead skewing first timed iteration
    _ = analyze_single_iteration(N, random.choice(bases))
    
    success_count = total_ops = 0
    start = time.perf_counter()
    for _ in range(num_iterations):
        r = analyze_single_iteration(N, random.choice(bases))
        total_ops += r['operations']
        if r['success']:
            success_count += 1
    elapsed = (time.perf_counter() - start) * 1000
    
    return {
        'N': N, 'num_bits': N.bit_length(), 'num_digits': len(str(N)),
        'factors': factorize(N), 'num_iterations': num_iterations,
        'success_count': success_count, 'success_rate': 100 * success_count / num_iterations,
        'total_operations': total_ops, 'avg_operations': total_ops / num_iterations,
        'total_time_ms': elapsed, 'avg_time_per_iteration_ms': elapsed / num_iterations
    }

def generate_semiprimes(min_n, max_n, num_samples):
    """Equally spaced semiprimes in [min_n, max_n]."""
    targets = np.linspace(min_n, max_n, num_samples)
    numbers = []
    for t in targets:
        start = max(15, int(t) | 1)  # Nearest odd ≥ 15
        for i in range(5000):  # Search outward for semiprime
            for c in [start + 2*i, max(15, start - 2*i)]:
                if is_semiprime(c):
                    numbers.append(c)
                    break
            else:
                continue
            break
        else:
            numbers.append(15)
    return sorted(set(numbers))

def extrapolate(results):
    """Fit ops = a×N^b; extrapolate to RSA key sizes."""
    N_arr = np.array([r['N'] for r in results])
    log_N, log_ops = np.log10(N_arr), np.log10(np.array([r['avg_operations'] for r in results]))
    slope, intercept = np.polyfit(log_N, log_ops, 1)  # log(ops) = slope*log(N) + intercept
    a = 10**intercept
    r2 = 1 - np.sum((log_ops - (slope*log_N + intercept))**2) / np.sum((log_ops - np.mean(log_ops))**2)  # Goodness of fit
    
    ref = results[-1]
    ref_time, ref_ops = ref['avg_time_per_iteration_ms']/1000, ref['avg_operations']
    sec_per_year = 365.25 * 24 * 3600
    # Extrapolate ops and time to RSA bit sizes
    rsa = {}
    for name, bits in [('RSA-512', 512), ('RSA-1024', 1024), ('RSA-2048', 2048), ('RSA-4096', 4096)]:
        log_N_rsa = bits * np.log10(2)
        log_ops_rsa = intercept + slope * log_N_rsa
        log_time = log_ops_rsa + np.log10(ref_time) - np.log10(ref_ops) if ref_ops else -9
        log_years = log_time - np.log10(sec_per_year)
        rsa[name] = {'ops': log_ops_rsa, 'years': log_years}
    
    return {'exponent': slope, 'coefficient': a, 'log_a': intercept, 'r_squared': r2, 'rsa': rsa,
            'fitted_log_ops': intercept + slope * log_N, 'log_N': log_N, 'log_ops': log_ops}

def main():
    MIN_N, MAX_N, NUM_SAMPLES, ITERATIONS = 15, 100_000_000, 1000, 10
    
    N_values = generate_semiprimes(MIN_N, MAX_N, NUM_SAMPLES)
    # Run Shor on each semiprime
    results = []
    start = time.time()
    
    for i, N in enumerate(N_values, 1):
        if i % 10 == 0 or i == 1:
            pct = 100 * i / len(N_values)
            eta = (time.time() - start) / i * (len(N_values) - i) if i else 0
            print(f"[{i}/{len(N_values)}] {pct:.1f}% | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERATIONS)
        if r:
            results.append(r)
    
    fit = extrapolate(results)  # Fit scaling law, extrapolate to RSA
    print(f"\nDone in {time.time()-start:.1f}s")
    print(f"Scaling: O(N^{fit['exponent']:.2f}), R² = {fit['r_squared']:.3f}")
    print(f"RSA-2048: ~10^{fit['rsa']['RSA-2048']['ops']:.0f} ops, ~10^{fit['rsa']['RSA-2048']['years']:.0f} years")
    
    # Export results and RSA extrapolation to Excel
    cwd = os.getcwd()
    excel_path = os.path.join(cwd, "Classical_Shor_Analysis_Results.xlsx") if os.path.basename(cwd) == "Comparison analysis circuits" else os.path.join(cwd, "Comparison analysis circuits", "Classical_Shor_Analysis_Results.xlsx")
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as w:
            # Sheet 1: Full results with log values and fitted line
            pd.DataFrame([{
                'N': r['N'],
                'Bits': r['num_bits'],
                'Digits': r['num_digits'],
                'Factors': 'x'.join(map(str, r['factors'])),
                'Success_Rate': r['success_rate'],
                'Total_Ops': r['total_operations'],
                'Avg_Ops': r['avg_operations'],
                'Total_Time_ms': round(r['total_time_ms'], 4),
                'Avg_Time_Per_Iteration_ms': round(r['avg_time_per_iteration_ms'], 4),
                'Log10_N': round(fit['log_N'][i], 4),
                'Log10_Avg_Ops': round(fit['log_ops'][i], 4),
                'Log10_Avg_Ops_Fitted': round(fit['fitted_log_ops'][i], 4),
                'Log10_Avg_Time_s': round(np.log10(r['avg_time_per_iteration_ms'] / 1000), 4) if r['avg_time_per_iteration_ms'] > 0 else None
            } for i, r in enumerate(results)]).to_excel(w, sheet_name='Results', index=False)
            
            # Sheet 2: RSA extrapolation
            pd.DataFrame([{
                'Key': k,
                'Bits': bits,
                'Log10_N': round(bits * np.log10(2), 1),
                'Log10_Ops': round(v['ops'], 1),
                'Log10_Years': round(v['years'], 1)
            } for (k, v), bits in zip(fit['rsa'].items(), [512, 1024, 2048, 4096])]).to_excel(w, sheet_name='RSA_Extrapolation', index=False)
            
            # Sheet 3: Fit parameters
            pd.DataFrame([{
                'Parameter': 'Exponent (b)',
                'Value': round(fit['exponent'], 4),
                'Description': 'Slope of log-log fit; ops scales as N^b'
            }, {
                'Parameter': 'Coefficient (a)',
                'Value': round(fit['coefficient'], 6),
                'Description': 'Multiplier in ops = a * N^b'
            }, {
                'Parameter': 'Log10(a)',
                'Value': round(fit['log_a'], 4),
                'Description': 'Y-intercept of log-log fit'
            }, {
                'Parameter': 'R_squared',
                'Value': round(fit['r_squared'], 4),
                'Description': 'Goodness of fit (1.0 = perfect)'
            }, {
                'Parameter': 'N_min',
                'Value': results[0]['N'],
                'Description': 'Smallest semiprime tested'
            }, {
                'Parameter': 'N_max',
                'Value': results[-1]['N'],
                'Description': 'Largest semiprime tested'
            }, {
                'Parameter': 'Num_Samples',
                'Value': len(results),
                'Description': 'Number of semiprimes tested'
            }, {
                'Parameter': 'Iterations_Per_Sample',
                'Value': ITERATIONS,
                'Description': 'Random bases tested per semiprime'
            }]).to_excel(w, sheet_name='Fit_Parameters', index=False)
            
        print(f"Exported: {excel_path}")
    except Exception as e:
        print(f"Excel export failed: {e}")

if __name__ == "__main__":
    main()

[1/1000] 0.1% | ETA: 0s
[10/1000] 1.0% | ETA: 88s
[20/1000] 2.0% | ETA: 144s
[30/1000] 3.0% | ETA: 222s
